In [0]:
%python
dbutils.widgets.text("caso", "monark", "Caso")
dbutils.widgets.text("versao_pipeline", "v0.2.0-dev", "Versao do pipeline")

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

-- =====================================================================
-- =====================================================================

CREATE OR REPLACE VIEW silver.v_normalizado
COMMENT 'Une as duas fontes de ingestão num único formato. É o normalizador do pipeline.'
AS
WITH graphql AS (
  SELECT
    a.arquivo_id, a.caso_slug, r.linha,
    get_json_object(r.payload,'$.legacy.id_str')                                AS id_nativo,
    get_json_object(r.payload,'$.core.user_results.result.core.screen_name')    AS autor_handle,
    get_json_object(r.payload,'$.core.user_results.result.rest_id')             AS autor_id_nativo,
    to_timestamp(substring(get_json_object(r.payload,'$.legacy.created_at'), 5),
                 'MMM dd HH:mm:ss Z yyyy')                                  AS created_at,
    get_json_object(r.payload,'$.legacy.full_text')                             AS texto,
    get_json_object(r.payload,'$.legacy.lang')                                  AS idioma,
    get_json_object(r.payload,'$.legacy.in_reply_to_status_id_str')             AS ref_status_id,
    get_json_object(r.payload,'$.legacy.in_reply_to_screen_name')               AS ref_handle,
    CAST(get_json_object(r.payload,'$.legacy.is_quote_status') AS BOOLEAN)      AS eh_quote,
    CAST(get_json_object(r.payload,'$.legacy.favorite_count') AS INT)           AS likes,
    CAST(get_json_object(r.payload,'$.legacy.retweet_count')  AS INT)           AS retweets,
    CAST(get_json_object(r.payload,'$.legacy.quote_count')    AS INT)           AS quotes,
    CAST(get_json_object(r.payload,'$.legacy.reply_count')    AS INT)           AS respostas,
    transform(
      from_json(get_json_object(r.payload,'$.legacy.entities.user_mentions'),
                'array<struct<id_str:string, screen_name:string>>'),
      m -> named_struct('id_nativo', m.id_str, 'handle', lower(m.screen_name))
    )                                                                           AS mencoes,
    transform(
      from_json(get_json_object(r.payload,'$.legacy.entities.hashtags'),
                'array<struct<text:string>>'),
      h -> lower(h.text)
    )                                                                           AS hashtags,
    CAST(NULL AS STRING)                                                        AS stance_previa,
    'graphql'                                                                   AS fonte
  FROM bronze.registro r
  JOIN bronze.arquivo  a USING (arquivo_id)
  WHERE a.fonte = 'graphql'
),
consolidado AS (
  SELECT
    a.arquivo_id, a.caso_slug, r.linha,
    get_json_object(r.payload,'$.id')                                           AS id_nativo,
    get_json_object(r.payload,'$.user')                                         AS autor_handle,
    CAST(NULL AS STRING)                                                        AS autor_id_nativo,
    to_timestamp(get_json_object(r.payload,'$.created_at_iso'))                 AS created_at,
    get_json_object(r.payload,'$.text')                                         AS texto,
    CAST(NULL AS STRING)                                                        AS idioma,
    nullif(get_json_object(r.payload,'$.in_reply_to_status_id'),'')             AS ref_status_id,
    nullif(get_json_object(r.payload,'$.in_reply_to_user'),'')                  AS ref_handle,
    CAST(get_json_object(r.payload,'$.is_quote') AS BOOLEAN)                    AS eh_quote,
    CAST(get_json_object(r.payload,'$.like_count')    AS INT)                   AS likes,
    CAST(get_json_object(r.payload,'$.retweet_count') AS INT)                   AS retweets,
    CAST(get_json_object(r.payload,'$.quote_count')   AS INT)                   AS quotes,
    CAST(get_json_object(r.payload,'$.reply_count')   AS INT)                   AS respostas,
    transform(
      from_json(get_json_object(r.payload,'$.mentions'),
                'array<struct<id_str:string, username:string>>'),
      m -> named_struct('id_nativo', m.id_str, 'handle', lower(m.username))
    )                                                                           AS mencoes,
    transform(
      from_json(get_json_object(r.payload,'$.hashtags'), 'array<string>'),
      h -> lower(h)
    )                                                                           AS hashtags,
    nullif(get_json_object(r.payload,'$.stance'),'')                            AS stance_previa,
    'consolidado'                                                               AS fonte
  FROM bronze.registro r
  JOIN bronze.arquivo  a USING (arquivo_id)
  WHERE a.fonte = 'consolidado'
),
uniao AS (SELECT * FROM graphql UNION ALL SELECT * FROM consolidado)
SELECT
  *,
  CASE
    WHEN ref_status_id IS NOT NULL OR ref_handle IS NOT NULL THEN 'reply'
    WHEN eh_quote                                            THEN 'quote'
    ELSE 'original'
  END AS tipo_ref
FROM uniao;

-- =====================================================================
-- =====================================================================

CREATE OR REPLACE VIEW silver.v_deduplicado
COMMENT 'Uma linha por postagem. Contadores = máximo observado entre as capturas do mesmo id.'
AS
SELECT
  caso_slug, id_nativo,
  min_by(autor_handle,    linha) AS autor_handle,
  min_by(autor_id_nativo, linha) AS autor_id_nativo,
  min_by(created_at,      linha) AS created_at,
  min_by(texto,           linha) AS texto,
  min_by(idioma,          linha) AS idioma,
  min_by(ref_status_id,   linha) AS ref_status_id,
  min_by(ref_handle,      linha) AS ref_handle,
  min_by(tipo_ref,        linha) AS tipo_ref,
  min_by(mencoes,         linha) AS mencoes,
  min_by(hashtags,        linha) AS hashtags,
  min_by(stance_previa,   linha) AS stance_previa,
  min_by(fonte,           linha) AS fonte,
  max(likes)     AS likes,
  max(retweets)  AS retweets,
  max(quotes)    AS quotes,
  max(respostas) AS respostas,
  count(*)       AS capturas
FROM silver.v_normalizado
GROUP BY caso_slug, id_nativo;



In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

DELETE FROM silver.qc_resultado
WHERE caso_slug = :caso AND versao_pipeline = :versao_pipeline;

INSERT INTO silver.qc_resultado (caso_slug, indicador, valor, severidade, aprovado, detalhe, versao_pipeline, verificado_em)
WITH base  AS (SELECT * FROM silver.v_normalizado WHERE caso_slug = :caso),
     dedup AS (SELECT * FROM silver.v_deduplicado WHERE caso_slug = :caso),
     arq   AS (SELECT * FROM bronze.arquivo       WHERE caso_slug = :caso)
SELECT q.*, current_timestamp() AS verificado_em FROM (
  SELECT :caso AS caso_slug, 'qc_linhas_bronze' AS indicador, CAST(count(*) AS DOUBLE) AS valor,
         'informa' AS severidade, true AS aprovado,
         'registros lidos da camada Bronze' AS detalhe, :versao_pipeline AS versao_pipeline
  FROM base
  UNION ALL
  SELECT :caso, 'qc_postagens_unicas', CAST(count(*) AS DOUBLE), 'informa', true,
         NULL, :versao_pipeline
  FROM dedup
  UNION ALL
  SELECT :caso, 'qc_duplicatas_removidas',
         CAST((SELECT count(*) FROM base) - (SELECT count(*) FROM dedup) AS DOUBLE),
         'informa', true, 'colapsadas por (caso, id_nativo)', :versao_pipeline
  UNION ALL
  SELECT :caso, 'qc_ids_duplicados', CAST(count(*) AS DOUBLE), 'informa', true,
         'ids que aparecem mais de uma vez no bruto', :versao_pipeline
  FROM dedup WHERE capturas > 1
  UNION ALL
  SELECT :caso, 'qc_duplicatas_divergentes', CAST(count(*) AS DOUBLE), 'alerta',
         count(*) = 0,
         'mesmo id com contadores diferentes: capturas em momentos distintos da raspagem', :versao_pipeline
  FROM (
    SELECT caso_slug, id_nativo FROM base
    GROUP BY caso_slug, id_nativo
    HAVING count(DISTINCT concat_ws('|', likes, retweets, quotes, respostas)) > 1
  )
  UNION ALL
  SELECT :caso, 'qc_id_fora_do_padrao', CAST(count(*) AS DOUBLE), 'bloqueia',
         count(*) = 0, 'id_nativo deve ser so digitos, 15 a 20', :versao_pipeline
  FROM dedup WHERE NOT (id_nativo RLIKE '^[0-9]{15,20}$')
  UNION ALL
  SELECT :caso, 'qc_descartadas_sem_autor', CAST(count(*) AS DOUBLE), 'alerta',
         count(*) = 0, 'postagens sem handle de autor, descartadas na promocao', :versao_pipeline
  FROM dedup WHERE autor_handle IS NULL OR trim(autor_handle) = ''
  UNION ALL
  SELECT :caso, 'qc_pct_sem_autor',
         CAST(100.0 * sum(CASE WHEN autor_handle IS NULL OR trim(autor_handle) = '' THEN 1 ELSE 0 END)
              / count(*) AS DOUBLE),
         'bloqueia',
         100.0 * sum(CASE WHEN autor_handle IS NULL OR trim(autor_handle) = '' THEN 1 ELSE 0 END)
              / count(*) <= 1.0,
         'acima de 1 pct sem autor indica falha de coleta', :versao_pipeline
  FROM dedup
  UNION ALL
  SELECT :caso, 'qc_data_nao_parseada', CAST(count(*) AS DOUBLE), 'bloqueia',
         count(*) = 0, 'created_at nulo apos conversao', :versao_pipeline
  FROM dedup WHERE created_at IS NULL
  UNION ALL
  SELECT :caso, 'qc_fora_da_janela', CAST(count(*) AS DOUBLE), 'alerta',
         count(*) = 0, 'postagem fora do periodo declarado na coleta', :versao_pipeline
  FROM dedup d WHERE NOT EXISTS (
    SELECT 1 FROM arq a
    WHERE date(d.created_at) BETWEEN a.periodo_inicio AND a.periodo_fim)
  UNION ALL
  SELECT :caso, 'qc_data_no_futuro', CAST(count(*) AS DOUBLE), 'bloqueia',
         count(*) = 0, NULL, :versao_pipeline
  FROM dedup WHERE created_at > current_timestamp()
  UNION ALL
  SELECT :caso, 'qc_reply_sem_destino', CAST(count(*) AS DOUBLE), 'bloqueia',
         count(*) = 0, 'reply precisa de ref_handle', :versao_pipeline
  FROM dedup WHERE tipo_ref = 'reply' AND (ref_handle IS NULL OR trim(ref_handle) = '')
  UNION ALL
  SELECT :caso, 'qc_replies_reclassificados', CAST(count(*) AS DOUBLE), 'informa', true,
         'respostas que o corpus rotulava como original e o pipeline corrigiu', :versao_pipeline
  FROM dedup WHERE tipo_ref = 'reply'
  UNION ALL
  SELECT :caso, 'qc_contador_negativo', CAST(count(*) AS DOUBLE), 'bloqueia',
         count(*) = 0, NULL, :versao_pipeline
  FROM dedup WHERE likes < 0 OR retweets < 0 OR quotes < 0 OR respostas < 0
  UNION ALL
  SELECT :caso, 'qc_texto_vazio', CAST(count(*) AS DOUBLE), 'alerta',
         count(*) = 0, NULL, :versao_pipeline
  FROM dedup WHERE texto IS NULL OR trim(texto) = ''
  UNION ALL
  SELECT :caso, 'qc_idioma_inesperado', CAST(count(*) AS DOUBLE), 'alerta',
         count(*) = 0, 'postagem em idioma diferente do declarado apesar do filtro lang', :versao_pipeline
  FROM dedup WHERE idioma IS NOT NULL AND idioma <> 'pt'
  UNION ALL
  SELECT :caso, 'qc_mencoes_totais',
         CAST(sum(CASE WHEN mencoes IS NULL THEN 0 ELSE size(mencoes) END) AS DOUBLE),
         'informa', true, 'arestas do grafo de mencoes', :versao_pipeline
  FROM dedup
  UNION ALL
  SELECT :caso, 'qc_pct_mencoes_com_id',
         CAST(100.0 * sum(CASE WHEN m.id_nativo IS NOT NULL THEN 1 ELSE 0 END) / nullif(count(*), 0) AS DOUBLE),
         'alerta',
         sum(CASE WHEN m.id_nativo IS NULL THEN 1 ELSE 0 END) = 0,
         'mencoes com id nativo resolvem CONTA sem heuristica de handle', :versao_pipeline
  FROM dedup LATERAL VIEW explode(mencoes) t AS m
  UNION ALL
  SELECT :caso, 'qc_cv_volume_diario', CAST(stddev_samp(n) / avg(n) AS DOUBLE), 'informa', true,
         'coeficiente de variacao do volume diario', :versao_pipeline
  FROM (SELECT date(created_at) AS d, count(*) AS n FROM dedup GROUP BY 1)
  UNION ALL
  SELECT :caso, 'qc_com_stance_previa', CAST(count(*) AS DOUBLE), 'informa', true,
         'rotulos de versao anterior, sem confianca registrada', :versao_pipeline
  FROM dedup WHERE stance_previa IS NOT NULL
  UNION ALL
  SELECT :caso, 'qc_stance_previa_fora_do_dominio', CAST(count(*) AS DOUBLE), 'bloqueia',
         count(*) = 0, 'stance deve ser acusador, defensor ou neutro', :versao_pipeline
  FROM dedup WHERE stance_previa IS NOT NULL
    AND stance_previa NOT IN ('acusador', 'defensor', 'neutro')
) q;

CREATE OR REPLACE VIEW silver.v_qc_portao AS
SELECT caso_slug, versao_pipeline,
       sum(CASE WHEN severidade = 'bloqueia' AND NOT aprovado THEN 1 ELSE 0 END) AS bloqueios,
       sum(CASE WHEN severidade = 'alerta'   AND NOT aprovado THEN 1 ELSE 0 END) AS alertas,
       sum(CASE WHEN severidade = 'bloqueia' AND NOT aprovado THEN 1 ELSE 0 END) = 0 AS pode_promover
FROM silver.qc_resultado
GROUP BY caso_slug, versao_pipeline;

SELECT indicador, valor, severidade, aprovado, detalhe
FROM silver.qc_resultado
WHERE caso_slug = :caso AND versao_pipeline = :versao_pipeline
ORDER BY CASE severidade WHEN 'bloqueia' THEN 1 WHEN 'alerta' THEN 2 ELSE 3 END, indicador;


In [0]:
SELECT * FROM silver.v_qc_portao ORDER BY caso_slug;

In [0]:
-- Portao de QC como condicao de execucao (plano C, 16/09/2026):
-- se pode_promover for falso ou inexistente para (caso, versao_pipeline), a celula falha e a tarefa do job para aqui.
SELECT assert_true(
  (SELECT pode_promover FROM silver.v_qc_portao
   WHERE caso_slug = :caso AND versao_pipeline = :versao_pipeline),
  concat('portao de QC reprovou o caso ', :caso, ' na versao ', :versao_pipeline)
) AS portao_ok;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

CREATE OR REPLACE VIEW silver.v_promovivel (
  caso_slug COMMENT 'episodio a que o arquivo pertence',
  id_nativo, autor_handle, autor_id_nativo, created_at, texto, idioma,
  ref_status_id, ref_handle, tipo_ref, mencoes, hashtags, stance_previa, fonte,
  likes, retweets, quotes, respostas, capturas,
  texto_limpo COMMENT 'texto limpo pela regra canonica (claude/especificacao-limpeza-texto.md), a mesma expressao validada em 00_validacao_limpeza_texto (600/600)'
)
COMMENT 'VIEW: linhas de v_deduplicado que passam o QC (silver.v_qc_portao.pode_promover) e ja trazem texto_limpo calculado. Entrada dos MERGE/INSERT da Silver. Reescrita em 16/09/2026 (plano C, passo 1.1): a definicao anterior nao aplicava o portao nem calculava texto_limpo.'
AS
SELECT
  d.caso_slug, d.id_nativo, d.autor_handle, d.autor_id_nativo, d.created_at, d.texto, d.idioma,
  d.ref_status_id, d.ref_handle, d.tipo_ref, d.mencoes, d.hashtags, d.stance_previa, d.fonte,
  d.likes, d.retweets, d.quotes, d.respostas, d.capturas,
  trim(regexp_replace(
    regexp_replace(
      regexp_replace(
        regexp_replace(
          regexp_replace(d.texto, '\\p{Z}', ' '),
          'http\\S+|www\\S+|pic\\.twitter\\.com\\S+', ''),
        '@([A-Za-z0-9_]{1,15})', ' '),
      '#[\\p{L}\\p{N}_]+', ' '),
    '\\s+', ' ')) AS texto_limpo
FROM silver.v_deduplicado d
WHERE d.autor_handle IS NOT NULL AND trim(d.autor_handle) <> ''
  AND EXISTS (
    SELECT 1 FROM silver.v_qc_portao q
    WHERE q.caso_slug = d.caso_slug AND q.pode_promover
  );

MERGE INTO silver.conta AS alvo
USING (
  SELECT handle, max(id_nativo) AS id_nativo FROM (
    SELECT lower(autor_handle) AS handle, autor_id_nativo AS id_nativo FROM silver.v_promovivel WHERE caso_slug = :caso
    UNION ALL
    SELECT m.handle, m.id_nativo FROM silver.v_promovivel LATERAL VIEW explode(mencoes) t AS m WHERE caso_slug = :caso AND m.handle IS NOT NULL
    UNION ALL
    SELECT lower(ref_handle), NULL FROM silver.v_promovivel WHERE caso_slug = :caso AND ref_handle IS NOT NULL
  ) GROUP BY handle
) AS origem
ON alvo.plataforma = 'x' AND alvo.handle = origem.handle
WHEN MATCHED AND alvo.id_nativo IS NULL AND origem.id_nativo IS NOT NULL THEN UPDATE SET alvo.id_nativo = origem.id_nativo
WHEN NOT MATCHED THEN INSERT (plataforma, handle, id_nativo, criado_em) VALUES ('x', origem.handle, origem.id_nativo, current_timestamp());

SELECT COUNT(*) AS contas FROM silver.conta;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

MERGE INTO silver.postagem AS alvo
USING (
  SELECT d.id_nativo, d.caso_slug, c.conta_id AS autor_conta_id, d.created_at, d.texto, d.texto_limpo, d.idioma, d.tipo_ref, d.ref_status_id,
         cr.conta_id AS ref_conta_id, d.fonte
  FROM silver.v_promovivel d
  JOIN silver.conta c ON c.plataforma='x' AND c.handle = lower(d.autor_handle)
  LEFT JOIN silver.conta cr ON cr.plataforma='x' AND cr.handle = lower(d.ref_handle)
  WHERE d.caso_slug = :caso
) AS origem
ON alvo.plataforma = 'x' AND alvo.id_nativo = origem.id_nativo
WHEN NOT MATCHED THEN INSERT (plataforma, id_nativo, caso_slug, autor_conta_id, created_at, texto, texto_limpo, idioma, tipo_ref, ref_id_nativo, ref_conta_id, situacao, fonte, carregada_em)
VALUES ('x', origem.id_nativo, origem.caso_slug, origem.autor_conta_id, origem.created_at, origem.texto, origem.texto_limpo, origem.idioma, origem.tipo_ref, origem.ref_status_id, origem.ref_conta_id, 'ativa', origem.fonte, current_timestamp());

SELECT COUNT(*) AS postagens FROM silver.postagem;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

-- Reexecucao do mesmo caso substitui os snapshots (MVP, plano C 16/09/2026); semantica por coleta fica para a v2.
DELETE FROM silver.captura
WHERE postagem_id IN (SELECT postagem_id FROM silver.postagem WHERE caso_slug = :caso);

INSERT INTO silver.captura (postagem_id, likes, retweets, quotes, respostas, capturas, capturado_em)
SELECT 
  p.postagem_id, 
  d.likes, 
  d.retweets, 
  d.quotes, 
  d.respostas, 
  d.capturas, 
  current_timestamp()
FROM silver.v_promovivel d
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = d.id_nativo
WHERE d.caso_slug = :caso;

SELECT COUNT(*) AS total_capturas FROM silver.captura;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

-- Reexecutavel por caso (plano C 16/09/2026)
DELETE FROM silver.mencao
WHERE postagem_id IN (SELECT postagem_id FROM silver.postagem WHERE caso_slug = :caso);

INSERT INTO silver.mencao (postagem_id, conta_id)
SELECT DISTINCT
  p.postagem_id,
  c.conta_id
FROM (
  SELECT d.id_nativo, mencao.handle as mention_handle
  FROM silver.v_promovivel d
  LATERAL VIEW explode(d.mencoes) tbl AS mencao
  WHERE d.caso_slug = :caso AND mencao.handle IS NOT NULL
) exploded
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = exploded.id_nativo
JOIN silver.conta c 
  ON c.plataforma='x' AND c.handle = exploded.mention_handle;

SELECT COUNT(*) AS total_mencoes FROM silver.mencao;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

-- Reexecutavel por caso (plano C 16/09/2026)
DELETE FROM silver.postagem_hashtag
WHERE postagem_id IN (SELECT postagem_id FROM silver.postagem WHERE caso_slug = :caso);

INSERT INTO silver.postagem_hashtag (postagem_id, hashtag)
SELECT DISTINCT
  p.postagem_id,
  exploded.hashtag
FROM (
  SELECT d.id_nativo, hashtag
  FROM silver.v_promovivel d
  LATERAL VIEW explode(d.hashtags) tbl AS hashtag
  WHERE d.caso_slug = :caso AND hashtag IS NOT NULL
) exploded
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = exploded.id_nativo;

SELECT COUNT(*) AS total_hashtags FROM silver.postagem_hashtag;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

-- Reexecutavel por caso, restrito a versao previa (plano C 16/09/2026); nunca toca os rotulos do modelo
DELETE FROM silver.classificacao
WHERE versao = 'previa-sem-confianca'
  AND postagem_id IN (SELECT postagem_id FROM silver.postagem WHERE caso_slug = :caso);

INSERT INTO silver.classificacao (postagem_id, esquema, rotulo, modelo, versao, confianca, classificado_em)
SELECT 
  p.postagem_id,
  'stance',
  d.stance_previa,
  'bertimbau-stance',
  'previa-sem-confianca',
  CAST(NULL AS DOUBLE),
  current_timestamp()
FROM silver.v_promovivel d
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = d.id_nativo
WHERE d.caso_slug = :caso 
  AND d.stance_previa IS NOT NULL;

SELECT COUNT(*) AS total_classificacoes FROM silver.classificacao;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

SELECT 'conta' AS tabela, COUNT(*) AS linhas FROM silver.conta
UNION ALL SELECT 'postagem', COUNT(*) FROM silver.postagem
UNION ALL SELECT 'captura', COUNT(*) FROM silver.captura
UNION ALL SELECT 'mencao', COUNT(*) FROM silver.mencao
UNION ALL SELECT 'postagem_hashtag', COUNT(*) FROM silver.postagem_hashtag
UNION ALL SELECT 'classificacao', COUNT(*) FROM silver.classificacao
ORDER BY tabela;